1. Loading candidate profiles from a parquet file
2. Converting each row into a clean profile string (skipped if `profile_text` already exists)
3. Embedding profile text (resumable via memmap checkpoint)
4. Creating Elasticsearch index and bulk-uploading profiles + embeddings
5. Verifying document count and kNN search

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: False


In [ ]:
import json
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
from dotenv import load_dotenv
from pathlib import Path
from tqdm.auto import tqdm

load_dotenv(Path('../.env'))
EMBED_API_KEY = os.environ.get('EMBED_API_KEY')
if not EMBED_API_KEY:
    raise ValueError('EMBED_API_KEY not set; add it to .env')

DATASET_PATH = '../data/candidate_profiles.parquet'  # already has profile_text
ID_COLUMN = 'id'
MAX_ROWS = None                                       # set an int for a smaller test run
EMBED_BATCH_SIZE = 32
EMBED_WORKERS = 4
EMBED_API_URL = 'https://qwen-embed-runpod.boliu.info/v1/embeddings'
EMBED_MODEL = 'Qwen/Qwen3-Embedding-0.6B'
EMBEDDING_DIM = 1024
EMBEDDINGS_PATH = Path('../data/candidate_embeddings.npy')
PROGRESS_PATH = Path('../data/embed_progress.json')
ES_URL = 'http://localhost:9200'
ES_INDEX = 'candidate_profiles'

In [10]:
# Use pyarrow directly: pd.read_parquet re-imports pandas extension types and can
# fail in a long-lived notebook kernel with "pandas.period already defined".
df = pq.read_table(DATASET_PATH).to_pandas()

print(f'Loaded rows: {len(df):,}')
print('Columns:', list(df.columns))


Loaded rows: 210,250


In [11]:
# if you want a smaller dataset for testing
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

print(f"Kept rows: {len(df):,}")

Kept rows: 1,000


In [12]:
def _clean(value) -> str:
    if value is None or pd.isna(value):
        return ''
    return ' '.join(str(value).strip().split())


def row_to_profile(row: pd.Series) -> str:
    """Format a candidate row as labeled profile text for embedding."""
    position = _clean(row.get('Position'))
    moreinfo = _clean(row.get('Moreinfo'))
    cv = _clean(row.get('CV'))
    looking_for = _clean(row.get('Looking For'))
    highlights = _clean(row.get('Highlights'))
    english_level = _clean(row.get('English Level'))
    primary_keyword = _clean(row.get('Primary Keyword'))

    if not cv or cv == moreinfo:
        bio, extra_cv = moreinfo, ''
    elif not moreinfo:
        bio, extra_cv = cv, ''
    else:
        bio = cv if len(cv) > len(moreinfo) else moreinfo
        extra_cv = cv if cv != bio else ''

    exp = row.get('Experience Years')
    if pd.notna(exp):
        years = float(exp)
        experience = f'{years:g} year' if years == 1 else f'{years:g} years'
    else:
        experience = ''

    sections = [
        f'Position: {position}' if position else '',
        f'Bio: {bio}' if bio else '',
        f'Looking for: {looking_for}' if looking_for else '',
        f'Highlights: {highlights}' if highlights else '',
        f'English level: {english_level}' if english_level else '',
        f'Experience: {experience}' if experience else '',
        f'Primary keyword: {primary_keyword}' if primary_keyword else '',
        f'CV: {extra_cv}' if extra_cv else '',
    ]
    return '\n'.join(section for section in sections if section)


In [14]:
# 2. Converting each row into a clean profile string (skip if already present)
if 'profile_text' not in df.columns:
    df['profile_text'] = df.apply(row_to_profile, axis=1)

profile_texts = df['profile_text'].tolist()
print(f'Profile texts ready: {len(profile_texts):,}')

In [15]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

headers = {
    'Authorization': f'Bearer {EMBED_API_KEY}',
    'Content-Type': 'application/json',
}

_thread_local = threading.local()


def get_session() -> requests.Session:
    session = getattr(_thread_local, 'session', None)
    if session is None:
        session = requests.Session()
        session.headers.update(headers)
        _thread_local.session = session
    return session


def embed_texts(texts: list[str]) -> list[list[float]]:
    response = get_session().post(
        EMBED_API_URL,
        json={'model': EMBED_MODEL, 'input': texts},
        timeout=300,
    )
    response.raise_for_status()
    data = sorted(response.json()['data'], key=lambda item: item['index'])
    return [item['embedding'] for item in data]


def load_completed_starts(total: int, batch_size: int) -> set[int]:
    if not PROGRESS_PATH.exists():
        return set()
    progress = json.loads(PROGRESS_PATH.read_text())
    if progress.get('total') != total:
        return set()
    if 'completed_starts' in progress:
        return set(progress['completed_starts'])
    next_offset = int(progress.get('next_offset', 0))
    return set(range(0, next_offset, batch_size))


def contiguous_offset(completed_starts: set[int], batch_size: int, total: int) -> int:
    offset = 0
    while offset < total and offset in completed_starts:
        offset += batch_size
    return offset


def save_progress(total: int, completed_starts: set[int], batch_size: int) -> None:
    PROGRESS_PATH.parent.mkdir(parents=True, exist_ok=True)
    PROGRESS_PATH.write_text(
        json.dumps(
            {
                'total': total,
                'next_offset': contiguous_offset(completed_starts, batch_size, total),
                'completed_starts': sorted(completed_starts),
            },
            indent=2,
        )
    )


def embed_batch(job: tuple[int, list[str]]) -> tuple[int, list[list[float]]]:
    start, chunk = job
    return start, embed_texts(chunk)


texts = df['profile_text'].tolist()
total = len(texts)
completed_starts = load_completed_starts(total, EMBED_BATCH_SIZE)
resume_offset = contiguous_offset(completed_starts, EMBED_BATCH_SIZE, total)

if not EMBEDDINGS_PATH.exists():
    emb = np.memmap(
        EMBEDDINGS_PATH,
        dtype=np.float32,
        mode='w+',
        shape=(total, EMBEDDING_DIM),
    )
else:
    emb = np.memmap(
        EMBEDDINGS_PATH,
        dtype=np.float32,
        mode='r+',
        shape=(total, EMBEDDING_DIM),
    )

pending_jobs = [
    (start, texts[start : start + EMBED_BATCH_SIZE])
    for start in range(0, total, EMBED_BATCH_SIZE)
    if start not in completed_starts
]

print(f'Embedding {total:,} profiles via {EMBED_API_URL}')
print(
    f'Model: {EMBED_MODEL}  |  batch size: {EMBED_BATCH_SIZE}  |  workers: {EMBED_WORKERS}'
)
print(f'Resuming from offset: {resume_offset:,}  |  pending batches: {len(pending_jobs):,}')

with ThreadPoolExecutor(max_workers=EMBED_WORKERS) as pool:
    futures = [pool.submit(embed_batch, job) for job in pending_jobs]
    for future in tqdm(as_completed(futures), total=len(futures), desc='API batches', unit='batch'):
        start, vectors = future.result()
        end = start + len(vectors)
        emb[start:end] = vectors
        emb.flush()
        completed_starts.add(start)
        save_progress(total, completed_starts, EMBED_BATCH_SIZE)

save_progress(total, set(range(0, total, EMBED_BATCH_SIZE)), EMBED_BATCH_SIZE)
candidate_embeddings = np.asarray(emb)
print(f'Done. Embeddings shape: {candidate_embeddings.shape}')

Embedding 1,000 profiles via https://qwen-embed-runpod.boliu.info/v1/embeddings
Model: Qwen/Qwen3-Embedding-0.6B  |  batch size: 8


API batches:   0%|          | 0/125 [00:00<?, ?batch/s]

Done. Embeddings shape: (1000, 1024)


In [ ]:
from elasticsearch import Elasticsearch, NotFoundError

es = Elasticsearch(ES_URL)
info = es.info()
print(f"Connected to Elasticsearch {info['version']['number']}")

try:
    es.indices.delete(index=ES_INDEX)
    print(f'Deleted existing index: {ES_INDEX}')
except NotFoundError:
    pass

es.indices.create(
    index=ES_INDEX,
    mappings={
        'properties': {
            'id': {'type': 'keyword'},
            'profile_text': {'type': 'text'},
            'position': {'type': 'text'},
            'english_level': {'type': 'keyword'},
            'experience_years': {'type': 'float'},
            'primary_keyword': {'type': 'keyword'},
            'cv_lang': {'type': 'keyword'},
            'embedding': {
                'type': 'dense_vector',
                'dims': EMBEDDING_DIM,
                'index': True,
                'similarity': 'cosine',
            },
        }
    },
)
print(f'Created index: {ES_INDEX}')

In [ ]:
from elasticsearch import helpers


def _field(row: pd.Series, name: str, default=''):
    value = row.get(name, default)
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return default
    return value


def generate_actions():
    for i in range(len(df)):
        row = df.iloc[i]
        yield {
            '_index': ES_INDEX,
            '_id': row[ID_COLUMN],
            '_source': {
                'id': row[ID_COLUMN],
                'profile_text': row['profile_text'],
                'position': _field(row, 'Position'),
                'english_level': _field(row, 'English Level'),
                'experience_years': _field(row, 'Experience Years', default=None),
                'primary_keyword': _field(row, 'Primary Keyword'),
                'cv_lang': _field(row, 'CV_lang'),
                'embedding': candidate_embeddings[i].tolist(),
            },
        }


success, errors = helpers.bulk(
    es,
    generate_actions(),
    chunk_size=500,
    request_timeout=120,
    raise_on_error=False,
)

es.indices.refresh(index=ES_INDEX)
print(f'Indexed documents: {success:,}')
if errors:
    print(f'Bulk errors: {len(errors)}')
    print(errors[:3])

In [ ]:
doc_count = es.count(index=ES_INDEX)['count']
print(f'Documents in {ES_INDEX}: {doc_count:,} (expected {len(df):,})')

query_text = 'senior python backend developer'
query_vec = embed_texts([query_text])[0]

results = es.search(
    index=ES_INDEX,
    knn={
        'field': 'embedding',
        'query_vector': query_vec,
        'k': 5,
        'num_candidates': 100,
    },
    _source=['id', 'position', 'primary_keyword', 'experience_years'],
)

print(f"\nTop matches for: {query_text!r}")
for hit in results['hits']['hits']:
    source = hit['_source']
    print(
        f"  score={hit['_score']:.4f}  "
        f"id={source['id']}  "
        f"position={source.get('position', '')!r}  "
        f"keyword={source.get('primary_keyword', '')!r}"
    )